# Pears trading 2

### Importing packages

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Importing data

In [15]:
df_bid = pd.read_parquet("../data/processed/eurusd_dukascopy_bid_201901.parquet")
df_ask = pd.read_parquet("../data/processed/eurusd_dukascopy_ask_201901.parquet")

print(df_bid.head(3))
print(df_ask.head(3))

                          datetime  symbol price_type    price  volume
0 2019-01-01 22:02:37.254000+00:00  EURUSD        BID  1.14598    3.75
1 2019-01-01 22:02:38.590000+00:00  EURUSD        BID  1.14599    2.25
2 2019-01-01 22:02:39.138000+00:00  EURUSD        BID  1.14599    3.75
                          datetime  symbol price_type    price  volume
0 2019-01-01 22:02:37.254000+00:00  EURUSD        ASK  1.14682    0.75
1 2019-01-01 22:02:38.590000+00:00  EURUSD        ASK  1.14682    0.75
2 2019-01-01 22:02:39.138000+00:00  EURUSD        ASK  1.14684    0.75


In [16]:
df_bid = df_bid.rename(columns={'price': 'bid_price', 'volume': 'bid_volume'})
df_ask = df_ask.rename(columns={'price': 'ask_price', 'volume': 'ask_volume'})

df_ticks = pd.merge_asof(
    df_ask[['datetime', 'ask_price', 'ask_volume']],
    df_bid[['datetime', 'bid_price', 'bid_volume']],
    on = 'datetime',
    direction = 'backward'
)

df_ticks = df_ticks.dropna()
df_ticks['mid_price'] = (df_ticks['bid_price'] + df_ticks['ask_price']) / 2
df_ticks['total_volume'] = df_ticks['bid_volume'] + df_ticks['ask_volume']
df_ticks['cum_vol'] = df_ticks['total_volume'].cumsum()

VOL_THRESHOLD = 1000

df_ticks['bar_id'] = (df_ticks['cum_vol'] // VOL_THRESHOLD).astype(int)

volume_bars = df_ticks.groupby('bar_id').agg(
    timestamp=('datetime', 'last'),
    open=('mid_price', 'first'),
    high=('mid_price', 'max'),
    low=('mid_price', 'min'),
    close=('mid_price', 'last'),
    volume=('total_volume', 'sum')
)

volume_bars = volume_bars.set_index('timestamp')

print(f"Success! Compressed {len(df_ticks)} ticks into {len(volume_bars)} Volume Bars.")
print(volume_bars.head(3))

Success! Compressed 3477166 ticks into 17621 Volume Bars.
                                      open      high       low     close  \
timestamp                                                                  
2019-01-01 22:24:38.142000+00:00  1.146400  1.146770  1.146215  1.146390   
2019-01-01 22:44:49.031000+00:00  1.146400  1.146535  1.146170  1.146330   
2019-01-01 22:51:49.051000+00:00  1.146335  1.146370  1.146235  1.146265   

                                   volume  
timestamp                                  
2019-01-01 22:24:38.142000+00:00   998.97  
2019-01-01 22:44:49.031000+00:00   998.88  
2019-01-01 22:51:49.051000+00:00  1000.39  
